# CDR Analytics Report

This notebook loads CDR (call detail record) Parquet data from S3, runs analytics (revenue by service and call type, customer usage, service metrics, churn risk), and displays the results in an in-notebook report.

## 1. Spark session (client mode)

In [ ]:
import os
import socket
from datetime import datetime
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, countDistinct, desc, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

s3_bucket = os.getenv("S3_BUCKET", "telecom-cdr-data")
cdr_prefix = "cdr-data"
s3_endpoint = os.getenv("S3_ENDPOINT", os.getenv("S3_ENDPOINT_URL", "minio-service.minio.svc.cluster.local:9000"))
s3_access_key = os.getenv("S3_ACCESS_KEY", os.getenv("AWS_ACCESS_KEY_ID", "minio"))
s3_secret_key = os.getenv("S3_SECRET_KEY", os.getenv("AWS_SECRET_ACCESS_KEY", "minio123"))

hostname = socket.gethostname()
IPAddr = socket.gethostbyname(hostname)
nb_prefix = os.getenv("NB_PREFIX", "")
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as f:
    current_namespace = f.readline().strip()

sparkConf = SparkConf()
sparkConf.setMaster("k8s://https://" + os.environ["KUBERNETES_SERVICE_HOST"] + ":443")
sparkConf.set("spark.kubernetes.authenticate.caCertFile", "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt")
sparkConf.set("spark.submit.deployMode", "client")
sparkConf.set("spark.kubernetes.container.image", "quay.io/opendatahub-contrib/pyspark:s3.3.1-h3.3.4_v0.1.1")
sparkConf.set("spark.pyspark.python", "3")
sparkConf.set("spark.pyspark.driver.python", "3")
sparkConf.set("spark.kubernetes.namespace", current_namespace)
sparkConf.set("spark.driver.blockManager.port", "7777")
sparkConf.set("spark.driver.host", IPAddr)
sparkConf.set("spark.driver.port", "2222")
sparkConf.set("spark.driver.bindAddress", "0.0.0.0")
sparkConf.set("spark.executor.instances", "3")
sparkConf.set("spark.executor.memory", "2g")
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.executor.memoryOverhead", "512m")
sparkConf.set("spark.driver.memoryOverhead", "512m")
sparkConf.set("spark.memory.fraction", "0.8")
sparkConf.set("spark.memory.storageFraction", "0.3")
sparkConf.set("spark.app.name", "CDR Analytics Latest")
if nb_prefix:
    sparkConf.set("spark.ui.proxyBase", nb_prefix + "/proxy/4040/")

s3_endpoint_no_proto = s3_endpoint.split("://")[-1] if "://" in s3_endpoint else s3_endpoint
sparkConf.set("spark.hadoop.fs.s3a.endpoint", s3_endpoint_no_proto)
sparkConf.set("spark.hadoop.fs.s3a.access.key", s3_access_key)
sparkConf.set("spark.hadoop.fs.s3a.secret.key", s3_secret_key)
sparkConf.set("spark.hadoop.fs.s3a.path.style.access", "true")
sparkConf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sparkConf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
sparkConf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
sc = spark.sparkContext

print(f"Bucket: {s3_bucket}")
print(f"Deploy mode: client (driver on {hostname}, namespace {current_namespace})")

## 2. Find latest parquet under cdr-data/year/month/day

In [ ]:
import boto3

endpoint_url = s3_endpoint if s3_endpoint.startswith("http") else f"http://{s3_endpoint}"
use_ssl = endpoint_url.strip().lower().startswith("https")
s3_client = boto3.client(
    "s3",
    endpoint_url=endpoint_url,
    aws_access_key_id=s3_access_key,
    aws_secret_access_key=s3_secret_key,
    use_ssl=use_ssl,
    verify=use_ssl,
    config=boto3.session.Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

prefix = cdr_prefix + "/"
paginator = s3_client.get_paginator("list_objects_v2")
parquet_objects = []
for page in paginator.paginate(Bucket=s3_bucket, Prefix=prefix):
    for obj in page.get("Contents") or []:
        k = obj["Key"]
        if k.endswith(".parquet"):
            parquet_objects.append({"Key": k, "LastModified": obj.get("LastModified")})

if not parquet_objects:
    raise SystemExit(f"No .parquet files under s3://{s3_bucket}/{prefix}")

latest = max(parquet_objects, key=lambda x: x["Key"])
latest_key = latest["Key"]
s3_path = f"s3a://{s3_bucket}/{latest_key}"
print(f"Latest parquet: {latest_key}")
print(f"Full path: {s3_path}")

## 3. Read latest file with schema → cdr_df

In [ ]:
cdr_schema = StructType([
    StructField("cdr_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("call_start_time", StringType(), True),
    StructField("call_end_time", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("call_type", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("cost", FloatType(), True),
    StructField("destination_number", StringType(), True),
    StructField("created_at", StringType(), True),
])

raw_df = spark.read.schema(cdr_schema).parquet(s3_path)
cdr_df = raw_df.select("cdr_id", "customer_id", "duration_seconds", "call_type", "service_type", "cost", "destination_number")
total_records = cdr_df.count()
print(f"Loaded {total_records} records from latest file")
cdr_df.show(5, truncate=False)

## 4. Run analytics (revenue, usage, service, churn)

In [ ]:
revenue_by_service = cdr_df.groupBy("service_type").agg(
    sum("cost").alias("total_revenue"),
    avg("cost").alias("avg_revenue_per_call"),
    count("*").alias("total_calls"),
    countDistinct("customer_id").alias("unique_customers")
).orderBy(desc("total_revenue"))

revenue_by_call_type = cdr_df.groupBy("call_type").agg(
    sum("cost").alias("total_revenue"),
    avg("cost").alias("avg_revenue_per_call"),
    count("*").alias("total_calls")
).orderBy(desc("total_revenue"))

cdr_with_usage = cdr_df.withColumn("call_duration_minutes", col("duration_seconds") / 60)
customer_usage = cdr_with_usage.groupBy("customer_id").agg(
    count("*").alias("total_calls"),
    sum("call_duration_minutes").alias("total_minutes"),
    sum("cost").alias("total_spent"),
    countDistinct("service_type").alias("service_types_used"),
    countDistinct("call_type").alias("call_types_used")
).orderBy(desc("total_spent"))

service_metrics = cdr_with_usage.groupBy("service_type").agg(
    count("*").alias("total_calls"),
    countDistinct("customer_id").alias("unique_customers"),
    sum("call_duration_minutes").alias("total_minutes"),
    avg("call_duration_minutes").alias("avg_call_duration"),
    sum("cost").alias("total_revenue"),
    avg("cost").alias("avg_cost_per_call")
).orderBy(desc("total_revenue"))

customer_activity = cdr_df.groupBy("customer_id").agg(
    count("*").alias("total_calls"),
    sum("cost").alias("total_spent"),
    countDistinct("service_type").alias("service_types_used"),
    countDistinct("call_type").alias("call_types_used")
)
churn_risk = customer_activity.withColumn("churn_risk_level",
    when(col("total_calls") < 5, "High")
    .when(col("total_calls") < 10, "Medium")
    .otherwise("Low")
)

print("Analytics: revenue_by_service, revenue_by_call_type, customer_usage, service_metrics, churn_risk")

## 5. Report (in-notebook)

In [ ]:
from IPython.display import display, HTML, Markdown

total_customers = cdr_df.select("customer_id").distinct().count()
total_revenue = cdr_df.agg(sum("cost")).collect()[0][0] or 0.0
avg_call_duration = cdr_df.agg(avg("duration_seconds")).collect()[0][0] or 0.0
high_risk_count = churn_risk.filter(col("churn_risk_level") == "High").count()
gen_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

summary_html = f"""
<div style="font-family: system-ui; max-width: 900px;">
  <h2 style="color: #007bff; border-bottom: 2px solid #007bff;">CDR Analytics Report (latest file only)</h2>
  <p style="color: #666;">Generated {gen_time} | Source: {latest_key}</p>
  <table style="width: 100%; border-collapse: collapse; margin: 1em 0;">
    <tr style="background: #f0f4f8;"><td style="padding: 10px; font-weight: bold;">Total records</td><td style="padding: 10px;">{total_records:,}</td></tr>
    <tr><td style="padding: 10px; font-weight: bold;">Unique customers</td><td style="padding: 10px;">{total_customers:,}</td></tr>
    <tr style="background: #f0f4f8;"><td style="padding: 10px; font-weight: bold;">Total revenue</td><td style="padding: 10px;">${total_revenue:,.2f}</td></tr>
    <tr><td style="padding: 10px; font-weight: bold;">Avg call duration</td><td style="padding: 10px;">{avg_call_duration:.1f} sec</td></tr>
    <tr style="background: #f0f4f8;"><td style="padding: 10px; font-weight: bold;">High churn risk (&lt;5 calls)</td><td style="padding: 10px;">{high_risk_count:,}</td></tr>
  </table>
</div>
"""
display(HTML(summary_html))

display(Markdown("### Revenue by service type"))
display(revenue_by_service.limit(15).toPandas())

display(Markdown("### Revenue by call type"))
display(revenue_by_call_type.limit(15).toPandas())

display(Markdown("### Top customers by spending"))
display(customer_usage.limit(15).toPandas())

display(Markdown("### Service metrics"))
display(service_metrics.limit(15).toPandas())

display(Markdown("### Churn risk summary (by level)"))
churn_summary_df = churn_risk.groupBy("churn_risk_level").count().orderBy(desc("count"))
display(churn_summary_df.toPandas())

display(Markdown("### High-risk customers (sample)"))
display(churn_risk.filter(col("churn_risk_level") == "High").limit(15).toPandas())

## 6. Upload report to S3 (CSV)

In [ ]:
import io

report_prefix = os.getenv("CDR_REPORT_PREFIX", "cdr-report-csv")
run_id = datetime.now().strftime("%Y/%m/%d/%H%M%S")
uploaded = []

for name, df in [
    ("revenue_by_service", revenue_by_service),
    ("revenue_by_call_type", revenue_by_call_type),
    ("customer_usage", customer_usage),
    ("service_metrics", service_metrics),
    ("churn_risk", churn_risk),
]:
    csv_buf = io.StringIO()
    df.toPandas().to_csv(csv_buf, index=False)
    key = f"{report_prefix}/{run_id}/{name}.csv"
    s3_client.put_object(Bucket=s3_bucket, Key=key, Body=csv_buf.getvalue(), ContentType="text/csv")
    uploaded.append(key)

print(f"Uploaded {len(uploaded)} CSV(s) to s3://{s3_bucket}/{report_prefix}/{run_id}/")
for k in uploaded:
    print(f"  {k}")

## 7. Clean up – Stop Spark session

In [ ]:
print("Stopping Spark session...")
sc.stop()
print("Spark session stopped.")